In [ ]:
import pandas as pd

csv_path = r"C:\Users\OJASV\Desktop\DS_Practice\data-analysis\StudentsPerformance.csv"
df = pd.read_csv(csv_path)

print(df.head())

   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  


In [8]:
# Create average score
df["average_score"] = df[["math score", "reading score", "writing score"]].mean(axis=1)

# Create performance category
df["performance_category"] = pd.qcut(
    df["average_score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,average_score,performance_category
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667,Medium
1,female,group C,some college,standard,completed,69,90,88,82.333333,High
2,female,group B,master's degree,standard,none,90,95,93,92.666667,High
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333,Low
4,male,group C,some college,standard,none,76,78,75,76.333333,High


In [9]:
X = df.drop(columns=["average_score", "performance_category"])
y = df["performance_category"]

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

C:\Users\OJASV\AppData\Local\Temp\ipykernel_8620\2216326454.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline


preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

dt_model = Pipeline([
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(random_state=42))
])

dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

In [12]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

print("Decision Tree Results:\n")
print(classification_report(y_test, dt_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, dt_pred))

print("Accuracy:", accuracy_score(y_test, dt_pred))

Decision Tree Results:

              precision    recall  f1-score   support

        High       0.96      0.99      0.97        67
         Low       0.97      0.97      0.97        67
      Medium       0.95      0.92      0.94        66

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200

Confusion Matrix:
[[66  0  1]
 [ 0 65  2]
 [ 3  2 61]]
Accuracy: 0.96


In [13]:
preprocess_scaled = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

knn_model = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

In [14]:
print("k-NN Results:\n")
print(classification_report(y_test, knn_pred))

print("Accuracy:", accuracy_score(y_test, knn_pred))

k-NN Results:

              precision    recall  f1-score   support

        High       0.88      0.88      0.88        67
         Low       0.94      0.87      0.90        67
      Medium       0.76      0.82      0.79        66

    accuracy                           0.85       200
   macro avg       0.86      0.85      0.86       200
weighted avg       0.86      0.85      0.86       200

Accuracy: 0.855


In [15]:
nb_model = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", GaussianNB())
])

nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

In [16]:
print("Naive Bayes Results:\n")
print(classification_report(y_test, nb_pred))

print("Accuracy:", accuracy_score(y_test, nb_pred))

Naive Bayes Results:

              precision    recall  f1-score   support

        High       0.96      0.97      0.96        67
         Low       0.95      0.88      0.91        67
      Medium       0.86      0.91      0.88        66

    accuracy                           0.92       200
   macro avg       0.92      0.92      0.92       200
weighted avg       0.92      0.92      0.92       200

Accuracy: 0.92


In [17]:
cm = confusion_matrix(y_test, dt_pred)

accuracy = accuracy_score(y_test, dt_pred)
precision = precision_score(y_test, dt_pred, average="macro")
recall = recall_score(y_test, dt_pred, average="macro")
f1 = f1_score(y_test, dt_pred, average="macro")

print("Confusion Matrix:\n", cm)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix:
 [[66  0  1]
 [ 0 65  2]
 [ 3  2 61]]
Accuracy: 0.96
Precision: 0.9599319976205928
Recall: 0.9598221016131464
F1 Score: 0.9597330091623331


In [18]:
# Remove target variable
X_kmeans = df.drop(columns=["performance_category"])

preprocess_kmeans = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

X_transformed = preprocess_kmeans.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_transformed)

df["Cluster"] = clusters

In [19]:
print(df["Cluster"].value_counts())

print("\nCluster-wise Average Scores:")
print(df.groupby("Cluster")[["math score", "reading score", "writing score"]].mean())

Cluster
2    434
1    310
0    256
Name: count, dtype: int64

Cluster-wise Average Scores:
         math score  reading score  writing score
Cluster                                          
0         48.375000      51.027344      48.777344
1         81.274194      85.090323      84.351613
2         65.691244      68.497696      67.783410
